# День 2 на Kaggle: подбор lr и чистая абляция loss

Перед запуском (Settings справа):
- Accelerator: **GPU T4 x2** (используется одна карта, см. ячейку 1)
- Internet: **On**
- Add Input: приватный датасет с кодом (папки `src/` и `scripts/`) и приватный датасет со split (`split_ids.parquet`)

Работа в два прохода:
1. **Подбор lr** (`RUN_LR_SWEEP = True`, `RUN_FULL = False`): запусти ячейки по порядку, около 15 минут. Прочитай таблицу и выбери лучший lr.
2. **Полная абляция** (`RUN_LR_SWEEP = False`, `RUN_FULL = True`, `BEST_LR` = выбранное значение): Save Version -> Save & Run All (Commit), можно закрыть вкладку. Результаты во вкладке Output.

In [ ]:
import os, sys, glob, shutil

# Одна GPU: при 2 картах Trainer включит DataParallel и изменит эффективный batch size.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Кэш HF в /kaggle/temp, чтобы не раздувать /kaggle/working (там лимит и всё попадает в Output).
os.environ["HF_HOME"] = "/kaggle/temp/hf"

hits = glob.glob("/kaggle/input/**/data_utils.py", recursive=True)
assert hits, "Не найден src/data_utils.py. Подключи датасет с кодом (Add Input)."
SRC = os.path.dirname(hits[0])
ROOT_IN = os.path.dirname(SRC)
for need in ("scripts/04_ablation_loss.py", "scripts/05_lr_sweep.py", "src/train_utils.py"):
    assert os.path.exists(f"{ROOT_IN}/{need}"), f"В датасете с кодом нет {need}: загрузи новую версию датасета."

split_hits = glob.glob("/kaggle/input/**/split_ids.parquet", recursive=True)
assert split_hits, "Не найден split_ids.parquet. Подключи датасет со split."

PROJECT = "/kaggle/working/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/artifacts")
shutil.copytree(SRC, f"{PROJECT}/src")
shutil.copytree(f"{ROOT_IN}/scripts", f"{PROJECT}/scripts")
shutil.copy(split_hits[0], f"{PROJECT}/artifacts/split_ids.parquet")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

print("Код:", SRC)
print("Split:", split_hits[0])
print("Рабочая папка:", os.getcwd())

In [ ]:
!nvidia-smi -L
!pip list 2>/dev/null | grep -i -E "^(transformers|datasets|torch|scikit-learn) "

## Проверка split

Строки с сохранёнными `row_idx` должны иметь те же `id`, что при создании split. Если проверка не прошла, дальше не запускай.

In [ ]:
import pandas as pd
from src.data_utils import load_filtered

ds, label_names = load_filtered()
sp = pd.read_parquet("artifacts/split_ids.parquet")
assert len(ds) == 1_401_862, f"Размер после фильтрации {len(ds)}, ожидалось 1401862"
now_ids = ds.select(sp["row_idx"].tolist())["id"]
assert list(now_ids) == sp["id"].tolist(), "id не совпали: split на Kaggle отличается от локального"
print("Split совпадает с сохранённым:", sp["split"].value_counts().to_dict())

In [ ]:
RUN_LR_SWEEP = True     # проход 1: подбор learning rate (~15 минут)
RUN_FULL = False        # проход 2: полная абляция (2 ветки x 3 seed'а, 1.5-2 часа)
BEST_LR = 1e-4          # впиши значение, которое выбрал по таблице подбора

## Проход 1: подбор learning rate

Одна модель на каждое значение lr (обычная CE, один seed, 20k примеров, 1 эпоха). Скрипт сам печатает итоговую таблицу и подсказывает, не надо ли расширить сетку.

In [ ]:
if RUN_LR_SWEEP:
    !python scripts/05_lr_sweep.py --lrs 5e-5 1e-4 3e-4 1e-3

## Проход 2: полная абляция CE против weighted CE

Запускай после того, как выбрал `BEST_LR`. Для коммита поставь `RUN_LR_SWEEP = False`.

In [ ]:
if RUN_FULL:
    !python scripts/04_ablation_loss.py --n-train 200000 --epochs 2 --lr {BEST_LR} --seeds 777 778 779

In [ ]:
!ls -la results